<a href="https://colab.research.google.com/github/tensorbytes0202/Transformer-v-s-Bert-from-scratch-/blob/main/BERT_vs_Transformer_(Scratch_Study).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PHASE 1:


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv("/content/test.csv",
                    header=None,
                    names=["label","title","description"]
                    )
test = pd.read_csv(
    "test.csv",
    header=None,
    names=["label","title","description"]
)

print(train.shape)
print(test.shape)
train.head()

(7601, 3)
(7601, 3)


,label,title,description
0,Class Index,Title,Description
1,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
2,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
3,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
4,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...


In [3]:
train["text"] = (
    train["title"].astype(str)
    + " "
    + train["description"].astype(str)
)

test["text"] = (
    test["title"].astype(str)
    + " "
    + test["description"].astype(str)
)

train = train[["text","label"]]
test = test[["text","label"]]

train.head()

,text,label
0,Title Description,Class Index
1,Fears for T N pension after talks Unions repre...,3
2,The Race is On: Second Private Team Sets Launc...,4
3,Ky. Company Wins Grant to Study Peptides (AP) ...,4
4,Prediction Unit Helps Forecast Wildfires (AP) ...,4


In [6]:
train["length"]  =train["text"].apply(lambda x:len(str(x).split())
)

train["length"].describe()

,length
count,7601.000000
mean,37.716353
std,10.137051
min,2.000000
25%,32.000000
50%,37.000000
75%,43.000000
max,137.000000


In [7]:
train["length"].quantile(0.95)

np.float64(52.0)

In [10]:
import re

def tokenize(text):

    text = text.lower()

    text = re.sub(
        r'[^a-z0-9 ]',
        '',
        text
    )

    return text.split()

In [11]:
tokenize(
    "Hello World! AI is Amazing."
)

['hello', 'world', 'ai', 'is', 'amazing']

In [14]:
counter  = Counter()
for text in train["text"]:

  tokens = tokenize(text)
  counter.update(tokens)

word2idx = {
    "<PAD>":0,
    "<UNX":1
}

In [15]:
idx = 2

for word,count in counter.items():
  if count >=2:
    word2idx[word] = idx

    idx+=1

In [16]:
len(word2idx)

13670

In [18]:
def encode(text):

  tokens = tokenize(text)

  return [
      word2idx.get(token,1)
      for token in tokens
  ]

In [19]:
encode(
    "Artificial Intelligence"
)

[2935, 2458]

In [20]:
MAX_LEN  = 100

def pad(sequence):
  sequence = sequence[:MAX_LEN]

  sequence += [0] * (
      MAX_LEN - len(sequence)
  )

  return sequence

In [21]:
X = [
    pad(encode(text)
    )
    for text in train["text"]
]

y = train["label"].values

In [22]:
print(len(X))
print(len(X[0]))

7601
100


In [23]:
train["length"].describe()
train["length"].quantile(0.95)
len(word2idx)

13670

PHASE 2:


Hyper parameters

In [24]:
VOCAB_SIZE = len(word2idx)

EMBED_DIM = 128
NUM_HEADS = 4

FF_DIM = 512

NUM_LAYERS = 2

NUM_CLASSES = 4
MAX_LEN = 100

DROUPOUT  = 0.1

Embedding Layer

In [29]:
class TokenEmbedding(nn.Module):

  def __init__(self,vocab_size,embed_dim):

    super().__init__()

    self.embedding  = nn.Embedding(vocab_size,embed_dim,padding_idx=0)

  def forward(self,x):

    return self.embedding(x)

In [30]:
emb = TokenEmbedding(
    VOCAB_SIZE,
    EMBED_DIM
)

sample = torch.tensor(X[:2])

print(
    emb(sample).shape
)

torch.Size([2, 100, 128])


In [31]:
import math

Positional Encoding

In [36]:
class PositionalEncoding(nn.Module):

  def __init__(self,embed_dim,max_len=5000):

    super().__init__()

    pe = torch.zeros(max_len,embed_dim)

    position = torch.arange(0,max_len).unsqueeze(1)

    div_term = torch.exp(torch.arange(0,embed_dim,2)*(-math.log(10000.0)/ embed_dim)
        )

    pe[:,0::2] = torch.sin(position * div_term)

    pe[:,1::2]  =torch.cos(position * div_term)

    pe = pe.unsqueeze(0)

    self.register_buffer("pe",pe)

  def forward(self,x):
       return x + self.pe[:,:x.size(1)]

In [37]:
pos = PositionalEncoding(
    EMBED_DIM
)

out = pos(
    emb(sample)
)

print(out.shape)

torch.Size([2, 100, 128])


Multi-Head Attention


In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(
      self,
      embed_dim,
      num_heads
  ):

    super().__init__()

    self.embed_dim = embed_dim
    self.num_heads = num_heads
    self.head_dim = (embed_dim // num_heads)


